# LangChain List Output Parsers Reference

Developer-facing statements defined in `langchain_core.output_parsers.list`.

# `droplastn`

Yields every element of an iterator except its final `n` elements.

```python
droplastn(
    iter: Iterator[T], # Iterator whose trailing elements should be withheld
    n: int, # Number of trailing elements to drop
) -> Iterator[T] # Elements preceding the final n elements
```

The function buffers elements in a `deque` and yields an element whenever the buffer grows beyond `n`.

# `ListOutputParser: BaseTransformOutputParser[list[str]]`

Abstract base class for parsers that convert model output into lists of strings and support incremental list-item streaming.

## Required subclass hooks

### `parse`

Parses complete model output into list items.

```python
@abstractmethod
parse(
    self,
    text: str, # Model output to parse
) -> list[str] # Parsed list items
```

A concrete subclass must implement this method. The abstract method body does not explicitly raise `NotImplementedError`.

## Optional subclass hooks

### `parse_iter`

Returns regular-expression matches for list items in the supplied text.

```python
parse_iter(
    self,
    text: str, # Model output to scan
) -> Iterator[re.Match[str]] # Match objects whose first capture group contains each item
```

The default implementation raises `NotImplementedError`.

## Behaviour

The inherited synchronous and asynchronous transform interfaces use the streaming logic implemented by this module.

String chunks are appended to an internal buffer. For `BaseMessage` chunks, only string content is appended; messages with non-string content are ignored.

When `parse_iter()` is implemented, all matches except the final match are treated as complete and emitted as single-item lists. The unconsumed suffix remains buffered because the final match may still be incomplete.

When `parse_iter()` raises `NotImplementedError`, the parser repeatedly calls `parse()` on the current buffer, emits every parsed item except the last, and keeps the last item buffered.

After the input stream ends, every item returned by `parse()` for the remaining buffer is emitted.

In [ ]:
import re # Import regular-expression support
from collections.abc import AsyncIterator, Iterator # Import iterator type hints

from langchain_core.messages import AIMessageChunk # Import a real streaming message chunk
from langchain_core.output_parsers.list import ListOutputParser # Import the abstract list parser


class SemicolonListParser(ListOutputParser): # Create a concrete streaming list parser
    pattern: str = r"\s*([^;]+);?" # Match one item followed by an optional semicolon

    def parse(self, text: str) -> list[str]: # Implement complete-text parsing
        return [item.strip() for item in text.split(";") if item.strip()] # Split and clean all items

    def parse_iter(self, text: str) -> Iterator[re.Match[str]]: # Implement incremental matching
        return re.finditer(self.pattern, text) # Return matches with each item in group one


parser = SemicolonListParser() # Create the concrete parser

complete_result = parser.parse("Python; SQL; LangChain") # Parse complete text
print("Complete result:", complete_result) # Display the parsed list

string_chunks = iter([ # Create streamed string chunks
    "Python; SQ", # Provide one complete and one incomplete item
    "L; Lang", # Complete SQL and begin LangChain
    "Chain;", # Complete LangChain
]) # Finish creating the iterator

print("\nSynchronous streaming:") # Display a heading

for parsed_chunk in parser.transform(string_chunks): # Parse chunks synchronously
    print(parsed_chunk) # Display each completed item

message_chunks = iter([ # Create streamed message chunks
    AIMessageChunk(content="Java; Py"), # Provide the first message chunk
    AIMessageChunk(content="Spark; Mongo"), # Continue the output
    AIMessageChunk(content="DB;"), # Complete the last item
]) # Finish creating the iterator

print("\nMessage-chunk streaming:") # Display a heading

for parsed_chunk in parser.transform(message_chunks): # Parse message chunks
    print(parsed_chunk) # Display each completed item


async def generate_chunks() -> AsyncIterator[str]: # Define an asynchronous chunk source
    yield "Pandas; Num" # Yield the first partial chunk
    yield "Py; Mat" # Complete NumPy and begin Matplotlib
    yield "plotlib;" # Complete Matplotlib


print("\nAsynchronous streaming:") # Display a heading

async for parsed_chunk in parser.atransform(generate_chunks()): # Parse asynchronously in Jupyter
    print(parsed_chunk) # Display each completed item

# `CommaSeparatedListOutputParser: ListOutputParser`

Parses comma-separated model output into a list of strings.

## Methods

### `is_lc_serializable`

Reports that the parser supports LangChain serialization.

```python
@classmethod
is_lc_serializable(
    cls,
) -> bool # Always True
```

### `get_lc_namespace`

Returns the serialization namespace for the parser.

```python
@classmethod
get_lc_namespace(
    cls,
) -> list[str] # `["langchain", "output_parsers", "list"]`
```

### `get_format_instructions`

Returns instructions requesting comma-separated values.

```python
@override
get_format_instructions(
    self,
) -> str # Comma-separated output instructions
```

### `parse`

Parses comma-separated text.

```python
@override
parse(
    self,
    text: str, # Model output to parse
) -> list[str] # Parsed values
```

The method uses `csv.reader()` with `quotechar='"'`, `delimiter=","`, and `skipinitialspace=True`, then flattens all parsed rows. Quoted values can therefore contain commas.

If `csv.reader()` raises `csv.Error`, the method falls back to splitting on commas and stripping surrounding whitespace from each part.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser # Import the real comma-separated parser


parser = CommaSeparatedListOutputParser() # Create the parser

print("Format instructions:") # Display a heading
print(parser.get_format_instructions()) # Display the required output format

simple_text = "Python, SQL, LangChain" # Create normal comma-separated text
simple_result = parser.parse(simple_text) # Parse the text directly

print("\nSimple result:", simple_result) # Display the parsed values

quoted_text = '"New Delhi, India", Mumbai, Bengaluru' # Put a comma inside a quoted value
quoted_result = parser.parse(quoted_text) # Parse the quoted CSV-style text

print("Quoted result:", quoted_result) # Display the correctly preserved quoted value

multiline_text = "Python, SQL\nPandas, NumPy" # Create values across two rows
multiline_result = parser.parse(multiline_text) # Parse and flatten all rows

print("Multiline result:", multiline_result) # Display the flattened values

invoked_result = parser.invoke( # Parse through the runnable interface
    "Java, PySpark, MongoDB", # Provide model-output text
    config={"run_name": "parse_technology_list"}, # Name the parser run
)

print("Invoke result:", invoked_result) # Display the runnable result

async_result = await parser.ainvoke( # Parse asynchronously in Jupyter
    '"Machine Learning, AI", Data Engineering, Analytics', # Provide quoted text
    config={"run_name": "parse_async_list"}, # Name the asynchronous run
)

print("Async result:", async_result) # Display the asynchronous result

print("\nSerializable:", parser.is_lc_serializable()) # Display serialization support
print("LC namespace:", parser.get_lc_namespace()) # Display the LangChain namespace

# `NumberedListOutputParser: ListOutputParser`

Parses numbered list items.

## Fields

```python
pattern: str = r"\d+\.\s([^\n]+)" # Regular expression matching numbered list items
```

## Constructor

```python
NumberedListOutputParser(
    *,
    pattern: str = r"\d+\.\s([^\n]+)", # Regular expression matching numbered list items
) -> None
```

## Methods

### `get_format_instructions`

Returns instructions requesting a numbered list with one item per line.

```python
@override
get_format_instructions(
    self,
) -> str # Numbered-list output instructions
```

### `parse`

Returns every item captured by `pattern`.

```python
parse(
    self,
    text: str, # Model output to parse
) -> list[str] # Captured numbered-list items
```

The method delegates to `re.findall()`.

### `parse_iter`

Returns an iterator over numbered-list matches.

```python
@override
parse_iter(
    self,
    text: str, # Model output to scan
) -> Iterator[re.Match[str]] # Numbered-list matches
```

The method delegates to `re.finditer()`.

In [ ]:
from langchain_core.output_parsers import NumberedListOutputParser # Import the real numbered-list parser


parser = NumberedListOutputParser() # Create the parser with the default pattern

print("Format instructions:") # Display a heading
print(parser.get_format_instructions()) # Display the required numbered-list format

numbered_text = """1. Learn Python
2. Practice SQL
3. Build LangChain projects""" # Create numbered model output

parsed_items = parser.parse(numbered_text) # Parse all numbered items
print("\nParsed items:", parsed_items) # Display the parsed list

print("\nMatches from parse_iter():") # Display a heading

for match in parser.parse_iter(numbered_text): # Iterate over every regex match
    print("Full match:", match.group(0)) # Display the complete matched text
    print("Captured item:", match.group(1)) # Display the captured list item

invoked_result = parser.invoke( # Parse through the runnable interface
    "1. Java\n2. PySpark\n3. MongoDB", # Provide numbered model output
    config={"run_name": "parse_numbered_technologies"}, # Name the parser run
)

print("\nInvoke result:", invoked_result) # Display the runnable result

async_result = await parser.ainvoke( # Parse asynchronously in Jupyter
    "1. Pandas\n2. NumPy\n3. Matplotlib", # Provide numbered model output
    config={"run_name": "parse_async_numbered_list"}, # Name the asynchronous run
)

print("Async result:", async_result) # Display the asynchronous result

custom_parser = NumberedListOutputParser( # Create a custom-numbering parser
    pattern=r"\d+\)\s([^\n]+)", # Match forms such as 1) Item
)

custom_text = "1) Delhi\n2) Mumbai\n3) Bengaluru" # Create custom-formatted text
custom_result = custom_parser.parse(custom_text) # Parse using the custom pattern

print("Custom-pattern result:", custom_result) # Display the custom result

# `MarkdownListOutputParser: ListOutputParser`

Parses Markdown bullet-list items beginning with `-` or `*`.

## Fields

```python
pattern: str = r"^\s*[-*]\s([^\n]+)$" # Regular expression matching Markdown bullet items
```

## Constructor

```python
MarkdownListOutputParser(
    *,
    pattern: str = r"^\s*[-*]\s([^\n]+)$", # Regular expression matching Markdown bullet items
) -> None
```

## Methods

### `get_format_instructions`

Returns instructions requesting a Markdown list.

```python
@override
get_format_instructions(
    self,
) -> str # Markdown-list output instructions
```

### `parse`

Returns every Markdown list item captured by `pattern`.

```python
parse(
    self,
    text: str, # Model output to parse
) -> list[str] # Captured Markdown list items
```

The method delegates to `re.findall()` with `re.MULTILINE`.

### `parse_iter`

Returns an iterator over Markdown list-item matches.

```python
@override
parse_iter(
    self,
    text: str, # Model output to scan
) -> Iterator[re.Match[str]] # Markdown list-item matches
```

The method delegates to `re.finditer()` with `re.MULTILINE`.

In [ ]:
from langchain_core.output_parsers import MarkdownListOutputParser # Import the real Markdown-list parser


parser = MarkdownListOutputParser() # Create the parser with the default pattern

print("Format instructions:") # Display a heading
print(parser.get_format_instructions()) # Display the required Markdown-list format

markdown_text = """- Learn Python
* Practice SQL
- Build LangChain projects""" # Create Markdown bullet-list output

parsed_items = parser.parse(markdown_text) # Parse all Markdown list items
print("\nParsed items:", parsed_items) # Display the parsed list

print("\nMatches from parse_iter():") # Display a heading

for match in parser.parse_iter(markdown_text): # Iterate over every regex match
    print("Full match:", match.group(0)) # Display the complete matched line
    print("Captured item:", match.group(1)) # Display the captured list item

invoked_result = parser.invoke( # Parse through the LangChain runnable interface
    "- Java\n* PySpark\n- MongoDB", # Provide Markdown list output
    config={"run_name": "parse_markdown_technologies"}, # Name the parser run
)

print("\nInvoke result:", invoked_result) # Display the runnable result

async_result = await parser.ainvoke( # Parse asynchronously in Jupyter
    "- Pandas\n* NumPy\n- Matplotlib", # Provide Markdown list output
    config={"run_name": "parse_async_markdown_list"}, # Name the asynchronous run
)

print("Async result:", async_result) # Display the asynchronous result

custom_parser = MarkdownListOutputParser( # Create a parser for plus-sign bullets
    pattern=r"^\s*\+\s([^\n]+)$", # Match lines such as + Item
)

custom_text = "+ Delhi\n+ Mumbai\n+ Bengaluru" # Create plus-sign bullet output
custom_result = custom_parser.parse(custom_text) # Parse using the custom pattern

print("Custom-pattern result:", custom_result) # Display the custom result